In [1]:
%load_ext autoreload
%autoreload 2

In [62]:
import json

from pathlib import Path
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import contextily as ctx

from pyproj import Transformer
import pygeohash as pgh
from pygeohash.viz import plot_geohash

import rasterio
import geopandas as gpd
import shapely
from shapely.geometry import Point

from dataset.dataset_meta_concat import SegmentationDatasetMetaConcat

In [3]:
data_path = Path("C:/Users/Administrator/PythonProjects/landcover_classification/ML_datasets/FLAIR1/flair_1_toy_dataset")
json_path = Path("C:/Users/Administrator/PythonProjects/landcover_classification/ML_datasets/FLAIR1/flair_1_metadata_aerial")

images_dir = data_path / "flair_1_toy_aerial_train"
labels_dir = data_path / "flair_1_toy_labels_train"

images = [p for p in images_dir.rglob("*.tif")]
labels = [p for p in labels_dir.rglob("*.tif")]
jsons = [p for p in json_path.rglob("*.json")]

print(len(images))
print(len(labels))
print(len(jsons))

200
200
1


#### Transform JSON, add spatial and temporal hashes

In [4]:
json_path = [p for p in jsons][0]

with open(json_path) as f:
    metadata = json.load(f)
# metadata

In [21]:
def add_lon_lat(metadata: dict) -> dict:
    new_metadata = {}
    transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)

    for key in metadata:
        item = metadata[key]
        x = item["patch_centroid_x"]
        y = item["patch_centroid_y"]
        lon, lat = transformer.transform(x, y)
        item["lon"] = lon
        item["lat"] = lat

        new_metadata[key] = item
    
    return new_metadata

    # # Optionally save back
    # with open("metadata_wgs84.json", "w") as f:
    #     json.dump(data, f, indent=2)
    
new_metadata = add_lon_lat(metadata)

In [22]:
def create_binary_geohash(
    latitude: float,
    longitude: float,
    precision: int = 12
    ):
    
    geohash = pgh.encode(latitude=latitude, longitude=longitude, precision=precision)
    
    # Convert back to the underlying binary string manually
    # Base32 char -> 5 bits based on geohash alphabet
    alphabet = "0123456789bcdefghjkmnpqrstuvwxyz"
    bitstr = "".join(f"{alphabet.index(c):05b}" for c in geohash)

    # Use that bitstr as a binary feature vector
    binary_gh = [int(b) for b in bitstr]
    
    return binary_gh, geohash

In [43]:
for k, v in new_metadata.items():
    lat = v["lat"]
    lon = v["lon"]
    binary_gh, geohash = create_binary_geohash(latitude=lat, longitude=lon, precision=7)
    v["geohash"] = geohash
    v["binary_geohash"] = np.array(binary_gh)

In [87]:
polygons = []

for k, val in new_metadata.items():
    geohash = val["geohash"]
    bbox = pgh.get_bounding_box(geohash)
    polygons.append(shapely.box(
        xmin = bbox.min_lat, 
        ymin = bbox.min_lon,
        xmax = bbox.max_lat,
        ymax = bbox.max_lon,
        # ccw=False
        ))
    

gdf_gh_bbox = gpd.GeoDataFrame(
    {
        "image": [f for f in new_metadata.keys()],
        "geohash": [f["geohash"] for f in new_metadata.values()]
    },
    geometry=polygons,
    crs="EPSG:4326"
)
save_dir = Path("C:/Users/Administrator/PythonProjects/landcover_classification/ML_datasets/FLAIR1/flair_1_toy_dataset/shapefiles")
save_path = save_dir / "geohashes_bbox.gpkg" 

gdf_gh_bbox.to_file(save_path, layer="geohashes", driver="GPKG")

In [ ]:

# Convert to Web Mercator for contextily
gdf_gh_bbox_3875 = gdf_gh_bbox.to_crs(epsg=3857)

# Plot
ax = gdf_gh_bbox_3875.plot(figsize=(8, 8), color="red", markersize=50)
# ax = gdf_gh_bbox.plot(figsize=(8, 8), color="red", markersize=50)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.show()

#### Test with Dataset

In [121]:
dataset = SegmentationDatasetMetaConcat(
    images_dir = data_path / "flair_1_toy_aerial_train",
    masks_dir = data_path / "flair_1_toy_labels_train_remap",
    meta_json_dir= json_path,
    augment = None,
    transform = None
)

In [127]:
dict = dataset.__getitem__(1)
print(dict["image_stem"])
print(dict["image"].shape)
print(dict["mask"].shape)

IMG_001119
torch.Size([5, 512, 512])
torch.Size([512, 512])
